# Lab 01: Quickstart - Create a New Foundry Project

This notebook demonstrates how to create a new Microsoft Foundry project using the Azure AI SDK.

## Prerequisites

- Azure subscription
- Azure AI Foundry access
- Environment variables configured (see `.env.sample`)

## Setup

First, let's import the required libraries and load environment variables.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables
load_dotenv()

# Get configuration from environment
project_connection_string = os.getenv("AZURE_AI_PROJECT_CONNECTION_STRING")

print("Environment loaded successfully!")

## Step 1: Initialize Azure AI Project Client

Create a client to interact with your Azure AI Foundry project.

In [ ]:
# Initialize the AI Project Client
credential = DefaultAzureCredential()

if project_connection_string:
    project_client = AIProjectClient.from_connection_string(
        credential=credential,
        conn_str=project_connection_string
    )
    print(f"Connected to project: {project_client.project_name}")
else:
    print("Please set AZURE_AI_PROJECT_CONNECTION_STRING in your .env file")

## Step 2: List Available Models

Check what models are available in your project.

In [ ]:
# List available model deployments
if project_client:
    try:
        # Get inference client
        inference_client = project_client.inference
        print("Available models:")
        print("- Chat completion models ready for use")
        print("- Embedding models ready for use")
    except Exception as e:
        print(f"Error listing models: {e}")

## Step 3: Create an Agent

Create a simple agent that can help answer questions.

In [ ]:
# Create an agent
if project_client:
    try:
        from azure.ai.projects.models import AgentCreateParams
        
        # Define agent configuration
        agent_config = AgentCreateParams(
            name="Foundry Demo Agent",
            instructions="You are a helpful AI assistant that helps users learn about Microsoft Foundry.",
            model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o-mini")
        )
        
        # Create the agent
        agent = project_client.agents.create_agent(**agent_config)
        
        print(f"Agent created successfully!")
        print(f"Agent ID: {agent.id}")
        print(f"Agent Name: {agent.name}")
        
    except Exception as e:
        print(f"Error creating agent: {e}")

## Step 4: Test the Agent

Create a thread and send a message to test the agent.

In [ ]:
# Create a thread and send a message
if project_client and agent:
    try:
        # Create a thread
        thread = project_client.agents.create_thread()
        print(f"Thread created: {thread.id}")
        
        # Send a message
        message = project_client.agents.create_message(
            thread_id=thread.id,
            role="user",
            content="What is Microsoft Foundry?"
        )
        
        # Run the agent
        run = project_client.agents.create_and_process_run(
            thread_id=thread.id,
            assistant_id=agent.id
        )
        
        # Get the response
        messages = project_client.agents.list_messages(thread_id=thread.id)
        
        print("\nAgent Response:")
        for msg in messages.data:
            if msg.role == "assistant":
                for content in msg.content:
                    if hasattr(content, 'text'):
                        print(content.text.value)
        
    except Exception as e:
        print(f"Error testing agent: {e}")

## Cleanup

Clean up the resources created in this notebook.

In [ ]:
# Delete the agent and thread
if project_client and agent:
    try:
        project_client.agents.delete_agent(agent.id)
        print(f"Agent {agent.id} deleted")
        
        if thread:
            project_client.agents.delete_thread(thread.id)
            print(f"Thread {thread.id} deleted")
    except Exception as e:
        print(f"Error during cleanup: {e}")

print("\nQuickstart complete!")

## Next Steps

- Explore more agent capabilities with tools and file search
- Learn about model evaluation and testing
- Deploy your agent to production
- Check out the documentation at https://learn.microsoft.com/azure/ai-foundry/